In [3]:
from pathlib import Path
import pickle
from treys import Card

import sys
import os

sys.path.append(
    os.path.abspath(os.path.join("..", "src", "game_abstraction"))
)

from board_features import extract_flop_board_features
from board_features import extract_turn_board_features

In [2]:
def load_flop_runtime(board_file,abstraction_file):
    with open(abstraction_file, "rb") as f:
        abstraction = pickle.load(f)

    with open(board_file, "rb") as f:
        bucket_data = pickle.load(f)

    return abstraction, bucket_data

In [3]:
def flop_to_bucket(card1, card2, card3, abstraction):
    vec = extract_flop_board_features(card1, card2, card3)
    vec_norm = (vec - abstraction["mean"]) / abstraction["std"]
    return int(abstraction["kmeans"].predict(vec_norm.reshape(1, -1))[0])

In [4]:
def get_flop_info(card1, card2, card3, abstraction, bucket_data):
    bucket = flop_to_bucket(card1, card2, card3, abstraction)

    info = bucket_data.get(bucket, None)
    if info is None:
        raise ValueError(f"No data for bucket {bucket}")

    return {
        "bucket": bucket,
        "range_metrics": info["range_metrics"],
        "hand_metrics": info["hand_metrics"]
    }

In [5]:
def load_turn_runtime(board_file,abstraction_file):
    with open(abstraction_file, "rb") as f:
        abstraction = pickle.load(f)

    with open(board_file, "rb") as f:
        bucket_data = pickle.load(f)

    return abstraction, bucket_data

In [12]:
def turn_to_bucket(card1, card2, card3,card4, abstraction):
    vec = extract_turn_board_features(card1, card2, card3,card4)
    vec_norm = (vec - abstraction["mean"]) / abstraction["std"]
    return int(abstraction["kmeans"].predict(vec_norm.reshape(1, -1))[0])

In [13]:
def get_turn_info(card1, card2, card3,card4, abstraction, bucket_data):
    bucket = turn_to_bucket(card1, card2, card3, card4, abstraction)

    info = bucket_data.get(bucket, None)
    if info is None:
        raise ValueError(f"No data for bucket {bucket}")

    return {
        "bucket": bucket,
        "range_metrics": info["range_metrics"],
        "hand_metrics": info["hand_metrics"]
    }

In [8]:
DATA_PATH = Path.cwd().parents[0] / "data"

abstraction, bucket_data = load_flop_runtime(DATA_PATH / "flop_bucket_metrics.pkl", DATA_PATH / "flop_abstraction.pkl")

info = get_flop_info(
    Card.new("As"),
    Card.new("Kd"),
    Card.new("Qc"),
    abstraction,
    bucket_data
)

print(info["bucket"])
print(info["hand_metrics"]['AKo'])
print(info["range_metrics"])

3
{'EHS': np.float64(0.7732087301587303), 'VAR': np.float64(0.13851550476190475), 'MED': np.float64(0.953125), 'IQR': np.float64(0.35868055555555556), 'NEG_POT': np.float64(0.21084682539682542), 'POS_POT': np.float64(0.7572642857142857), 'CRASH': np.float64(0.21084682539682542), 'DOM': np.float64(0.7572642857142857)}
{'RangeAdv': np.float64(0.03082109749225137), 'NutAdv': np.float64(0.029585798816568046), 'EPI': np.float64(0.08875739644970414), 'ECI': np.float64(0.18235575526463596), 'ShowdownDensity': np.float64(0.4911242603550296), 'LockIn': np.float64(0.8166247889170658)}


In [18]:
DATA_PATH = Path.cwd().parents[0] / "data"

abstraction, bucket_data = load_turn_runtime(DATA_PATH / "turn_bucket_metrics.pkl", DATA_PATH / "turn_abstraction.pkl")

info = get_turn_info(
    Card.new("As"),
    Card.new("Kd"),
    Card.new("Qc"),
    Card.new("Qd"),
    abstraction,
    bucket_data
)

print(info["bucket"])
print(info["hand_metrics"]['AA'])
print(info["range_metrics"])

18
{'EHS': np.float64(0.861111111111111), 'VAR': np.float64(0.10000000000000002), 'MED': np.float64(1.0), 'IQR': np.float64(0.1111111111111111), 'NEG_POT': np.float64(0.13333333333333333), 'POS_POT': np.float64(0.8555555555555555), 'CRASH': np.float64(0.13333333333333333), 'DOM': np.float64(0.8555555555555555)}
{'RangeAdv': np.float64(0.040164052471744816), 'NutAdv': np.float64(0.05325443786982249), 'EPI': np.float64(0.05325443786982249), 'ECI': np.float64(0.1459082250790626), 'ShowdownDensity': np.float64(0.5088757396449705), 'LockIn': np.float64(0.8485424376193607)}


In [1]:
def preflop_loader(preflop_file):
    with open(preflop_file, "rb") as f:
        file = pickle.load(f)
    return file

In [4]:
DATA_PATH = Path.cwd().parents[0] / "data"


In [5]:
data = preflop_loader(DATA_PATH / "preflop_metrics.pkl")
data

{'22': {'EHS': 0.50398,
  'VAR': 0.24524415959999998,
  'MED': 0.5,
  'IQR': 1.0,
  'NEG_POT': 0.48654,
  'POS_POT': 0.4945},
 '32s': {'EHS': 0.359315,
  'VAR': 0.2155152307749999,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.6113,
  'POS_POT': 0.32993},
 '32o': {'EHS': 0.32156,
  'VAR': 0.20262416639999997,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.64737,
  'POS_POT': 0.29049},
 '42s': {'EHS': 0.36476,
  'VAR': 0.21735014240000008,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.60652,
  'POS_POT': 0.33604},
 '42o': {'EHS': 0.33357,
  'VAR': 0.20699105510000002,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.63581,
  'POS_POT': 0.30295},
 '52s': {'EHS': 0.376535,
  'VAR': 0.21980389377499995,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.59356,
  'POS_POT': 0.34663},
 '52o': {'EHS': 0.34254,
  'VAR': 0.21008134839999998,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.62721,
  'POS_POT': 0.31229},
 '62s': {'EHS': 0.378245,
  'VAR': 0.22066321997500002,
  'MED': 0.0,
  'IQR': 1.0,
  'NEG_POT': 0.59273,
